# build_active_cve_set
`silver_vuln_alias`(`id_type == "CVE"`) → `active_cve_set`. "현재 Watchback이 EPSS 시계열을 관리하는 CVE 집합".

**알려진 구현 갭 보강** (04_Notebook_설계.md §5.3): 기존 설계는 매번 전체 overwrite만 하고 `first_seen_in_scope` 등 이력 컬럼이 없어 "어제와 오늘 비교"가 코드로 안 되는 문제가 있었다. 이 노트북은
- 기존 `active_cve_set`과 오늘자 alias 기준 CVE 집합을 **MERGE**해서 `first_seen_in_scope`/`last_seen_in_scope`/`discovered_by_batch_id`를 관리하고,
- 오늘 처음 발견된 CVE만 `stg_new_active_cve`에 남겨 `JOB_CVE_BACKFILL`이 그대로 소비하게 한다.

행을 삭제하지 않는다 — 오늘 alias에 일시적으로 안 잡히는 CVE도 이력을 보존하고, `last_seen_in_scope`로 최신성을 판단하게 한다.

In [ ]:
%run 00_silver_common

In [ ]:
alias_df = spark.table(TBL["vuln_alias"]).filter(F.col("id_type") == "CVE")

today_active = (
    alias_df
    .groupBy(F.col("id_value").alias("cve_id"))
    .agg(F.sort_array(F.collect_set("canonical_key")).alias("source_ghsa_ids"))
)

today_count = today_active.count()


In [ ]:
if table_exists(TBL["active_cve_set"]):
    existing = spark.table(TBL["active_cve_set"]).select(
        "cve_id", "first_seen_in_scope", "discovered_by_batch_id"
    )
    joined = today_active.join(existing, on="cve_id", how="left")

    merged = (
        joined
        .withColumn("first_seen_in_scope", F.coalesce(F.col("first_seen_in_scope"), F.lit(snapshot_date)))
        .withColumn("discovered_by_batch_id", F.coalesce(F.col("discovered_by_batch_id"), F.lit(batch_id)))
        .withColumn("last_seen_in_scope", F.lit(snapshot_date))
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("computed_at", F.lit(INGESTED_AT))
    )
    newly_active_df = joined.filter(
        F.col("first_seen_in_scope").isNull() | (F.col("discovered_by_batch_id") == batch_id)
    ).select("cve_id").distinct()
else:
    merged = (
        today_active
        .withColumn("first_seen_in_scope", F.lit(snapshot_date))
        .withColumn("last_seen_in_scope", F.lit(snapshot_date))
        .withColumn("discovered_by_batch_id", F.lit(batch_id))
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("computed_at", F.lit(INGESTED_AT))
    )
    newly_active_df = today_active.select("cve_id")


In [ ]:
merged.createOrReplaceTempView("_active_cve_merged")

if table_exists(TBL["active_cve_set"]):
    target = DeltaTable.forName(spark, TBL["active_cve_set"])
    (
        target.alias("t")
        .merge(spark.table("_active_cve_merged").alias("s"), "t.cve_id = s.cve_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    write_overwrite(merged, TBL["active_cve_set"])

write_overwrite(newly_active_df, TBL["stg_new_active_cve"])
newly_active_count = newly_active_df.count()

log_batch_run("build_active_cve_set", "SUCCESS", record_count=today_count)
log_dq_metric("build_active_cve_set", "newly_active_cve_count", newly_active_count, passed=True)
print(f"[build_active_cve_set] active={today_count} newly_active={newly_active_count} batch_id={batch_id}")
